# Prompt engineering

This page considers the approaches to improve the quality of the LLM outputs.

In [2]:
from langchain_ollama import ChatOllama

## Token sampling

At each step of the generation process, the model generates the next token by classifying it based on previous tokens. Thus, at some point, the model's predictions resemble the probabilities of the next token:

$\left(p_1, p_2, \ldots, p_n\right), \sum_{i=1}^n p_i =1$

Where $n$ is a vocabulary of the model.

**The temperature** regulates the randomness of the selected tokens. A value of 0 results in deterministic model outputs. The higher the value,a the more creative the model's output will be.

**Top-k** specifies the model that selects the next token from among the $k$ tokens with the highest probability.

**Top-p** it considers the smallest possible set of tokens whose cumulative probability exceeds a predefined threshold, $p$.

## Prompting techniques

There are different approaches associated to providing model information about the structure of required output:

- **Zero shot**: general prompting technique, just query to the model without providing any additional information.
- **One shot & Zero shot**: to explain the model the structure of the output you expect from it.

There are following options when specifying the general patterns of the model behaviour:

- **System prompting** sets the overall context and purpose for the language model.
- **Contextual prompting** provides specific details or background information relevant to the current convecrsation task.
- **Role prompting** assigning the a specific character or identity for the language model

**Step-back prompting**: Two prompts are provided: a spefic prompt and a more general prompt. The more general prompt ussually asks about typical approaches to the issue described in the specific prompt. The second prompt includes the model's answer to the general prompt as context and the specific prompt as the taks.

**Chain of Thought (CoT)**: A technique in which the model is asked to solve a task step by step. In the most basic implementation, the prompt literally asks the model to solve the problem "step by step".

## Chain of Thought

The chain of thought is a technique instructs the model to break down complex tasks into steps and complete them one by one.

---

The following cell applies a very small model to a simple task that requires to do one action before the other.

In [11]:
task = """
The cafeteria had 23 apples. If they used 20 to make lunch and bought 6
more, how many apples do they have?
"""

model = ChatOllama(model="llama3.2:1b", temperature=0)
ans = model.invoke(task).content
print(ans)

To find out how many apples the cafeteria has now, we need to add the number of apples they started with (23) to the number of apples they used (20) and then add the number of apples they bought (6).

First, let's add 23 and 20:
23 + 20 = 43

Now, let's add 6 to that result:
43 + 6 = 49

So, the cafeteria now has 49 apples.


Obviously the solution is not correct.

The following cell adds to the prompt CoT like instruction.

In [16]:
CoT_prompt = "\nDecompose task to steps and complete one after another."
print(model.invoke(task + CoT_prompt).content)

Here's the step-by-step solution:

**Step 1: Identify what we know**

* The cafeteria initially had 23 apples.
* They used 20 apples for lunch.
* After using some, they bought 6 more.

**Step 2: Calculate how many apples were left after lunch**

To find out how many apples were left after lunch, subtract the number of apples used from the initial number:

23 (initial apples) - 20 (apples used) = 3

So, there are 3 apples left in the cafeteria.

**Step 3: Add the new apples bought to the remaining apples**

Now, add the 6 new apples bought to the 3 apples that were already left:

3 (remaining apples) + 6 (new apples) = 9

Therefore, the cafeteria now has a total of 9 apples.


The model used more tokens to produce the output, but it produced the correct result.